# Live-consistent retrain (Tier 3)

Re-fits the AQI models so training and serving use the **same AQI definition** (the one the live ingest computes), then runs the
PM2.5 1h/6h audit/export and the 24h quantile / P(severe) export (decision D2).

| Section | What it does | Scored |
| :--- | :--- | :--- |
| A | 1h/6h LightGBM + 24h hybrid, recomputed AQI as features and target, frozen hyper-parameters and features | validation selects the iteration count only; **test is scored once** |
| B | PM2.5 1h/6h (and 24h) on live-masked inputs (no NOx/NO/solar, no ITO weather), Tier 2 variant selection on validation | test scored once, next to persistence |
| C | 24h quantile models + conformal pad + P(AQI >= 401) classifier, export only | as in Tier 2 |

**Before running:** upload `tier3_live_retrain_bundle.zip` to your Drive folder `aq_tier3_live` (or to `/content`), keep `SMOKE = False`,
then *Run all*. The notebook stops on any hash mismatch, any failed step, and on a second test scoring. It never edits the frozen files.

In [ ]:
SMOKE = False          # True = code-path check only (30 boosting rounds, 2 months of training, smoke profile); results must not be used
RUN_AQI, RUN_PM25, RUN_QUANTILE = True, True, True

In [ ]:
import os, sys, hashlib, subprocess, zipfile, pathlib, platform, json
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = pathlib.Path('/content/drive/MyDrive/aq_tier3_live')
except ImportError:                      # running locally
    PROJECT_DIR = pathlib.Path('.').resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE = next((p for p in [PROJECT_DIR / 'tier3_live_retrain_bundle.zip', pathlib.Path('/content/tier3_live_retrain_bundle.zip')] if p.exists()), None)
assert BUNDLE is not None, 'tier3_live_retrain_bundle.zip not found in the Drive folder or /content'
zipfile.ZipFile(BUNDLE).extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR); sys.path.insert(0, str(PROJECT_DIR))
print('project dir:', PROJECT_DIR, '| bundle:', BUNDLE)

In [ ]:
BUNDLE_SHA256 = {
 "Og Data/Delhi data/Anand Vihar, Delhi - DPCC/Raw Data/2024_raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H.csv": "83b841bac3b4e5edc2420e519f5810dc263a783ebc58abdbf37d7306506b01ea",
 "Og Data/Delhi data/Anand Vihar, Delhi - DPCC/Raw Data/2025_raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H.csv": "54d3037b3f77db940498e0be92fe705226a243884581e0018cd288ca8ec417f2",
 "Og Data/Delhi data/Bawana, Delhi - DPCC/Raw Data/2024_raw_data_hourly_bawana,_delhi_-_dpcc_1H.csv": "d710a69c6ead9398d191e61d22d3763db4e64e472a218d4b2b9c5991ba65ec1d",
 "Og Data/Delhi data/Bawana, Delhi - DPCC/Raw Data/2025_raw_data_hourly_bawana,_delhi_-_dpcc_1H.csv": "2af24b8f9d87a64cdd875abf4daf997edd4640d00210a46fbb4039b58e9b0231",
 "Og Data/Delhi data/Dwarka-Sector 8, Delhi - DPCC/Raw Data/2024_raw_data_hourly_dwarka-sector_8,_delhi_-_dpcc__1H.csv": "1c865139a3fd6a508b5f4556a534b7232b4b9fb7a3a5328093df2f50297eddfa",
 "Og Data/Delhi data/Dwarka-Sector 8, Delhi - DPCC/Raw Data/2025_raw_data_hourly_dwarka-sector_8,_delhi_-_dpcc__1H.csv": "0d984e8813edf3ea31344f2b0271bb71a95fb7983bb78ea6a1b25f61428261cb",
 "Og Data/Delhi data/ITO, Delhi - CPCB/Raw Data/2024_raw_data_hourly_ito,_delhi_-_cpcb_1H.csv": "ca8b455db7bbf5876cc0423ba5915bc0643e930fe0ed690a43e84fb1262bdfb7",
 "Og Data/Delhi data/ITO, Delhi - CPCB/Raw Data/2025_raw_data_hourly_ito,_delhi_-_cpcb_1H.csv": "937024113f1e211941e1fb180c8d0c095e228bf289b12dbca3711089506c88fb",
 "Og Data/Delhi data/Jahangirpuri, Delhi - DPCC\t/Raw Data/2024_raw_data_hourly_jahangirpuri,_delhi_-_dpcc_1H.csv": "f7373e98c579de8f47a7ae81dcef40b6138d6012ab1038c063ce8625941cb861",
 "Og Data/Delhi data/Jahangirpuri, Delhi - DPCC\t/Raw Data/2025_raw_data_hourly_jahangirpuri,_delhi_-_dpcc_1H.csv": "ef4c7558b353ecbda09cbfd608d7e4cf38f8931528f38b622657117d408985ad",
 "Og Data/Delhi data/Punjabi Bagh, Delhi - DPCC/Raw Data/2024_raw_data_hourly_punjabi_bagh,_delhi_-_dpcc_1H.csv": "6373a74ee8185cafcf0b81e1bb87a446231d4cc1a060d4de129937b7123e1aad",
 "Og Data/Delhi data/Punjabi Bagh, Delhi - DPCC/Raw Data/2025_raw_data_hourly_punjabi_bagh,_delhi_-_dpcc_1H.csv": "0ae394a69b9e96e4ecb647240ca92cc6965106eae7901f992bfe08bacce41252",
 "Og Data/Delhi data/R K Puram, Delhi - DPCC/Raw Data/2024_raw_data_hourly_r_k_puram,_delhi_-_dpcc_1H.csv": "6cd7c9f73942a117eee60ecabb1733637c42db627ff51a2cd0a670615df17877",
 "Og Data/Delhi data/R K Puram, Delhi - DPCC/Raw Data/2025_raw_data_hourly_r_k_puram,_delhi_-_dpcc_1H.csv": "983f2658bb6a08daac82ce0c4c14c451ca0f10ac1bc0941b66fd7562a7d2903a",
 "config/model_hashes.json": "110bb5cf94c7bab3fb35a5e71c476ccc3b95bb1b1abb111d61d15d8754f4d121",
 "etl/audit_ts.py": "6eb345c81a3298ff9bb3100b412d457472514b02eaa9bfaea02a7e833d80d7a5",
 "etl/audit_validation.py": "067c164b48a5d78f7154f4cb6a3d82beaf563c594d56ab01ff2d8b96e5ef1800",
 "etl/discover.py": "f24dec785bde954f7cfa6b326bafd3ee9d6e204a68137a5fe0892aa7c225b43f",
 "etl/export_kaggle_dataset.py": "dcdb83864043cfd15add8b2dacc0dfb3e9748ca92b44c57a57b7aa54f484e523",
 "etl/generate_mapping_report.py": "f9a8fb630e91bcf0a4ab8c6857cf9c7b22df163b734adfef0be3483d399b6675",
 "etl/load_postgres.py": "3baffa0b7b117ab8893ca66ceaceaf4c73b29327c1c75842e3d5021f77e2ec48",
 "etl/pipeline.py": "2ceafc703582e6a23072dc66bacdc2eadf32ffca8da2d185c2494914926c4522",
 "etl/run_full_load.py": "3403d20e4568f227191a933d6a398be5a177b084d77a58e2f69704abc0b26081",
 "etl/run_small_load.py": "71bb17db8454ad4935442f70233ce15445b1d1377ee9015856cd6e6b7637a42d",
 "etl/schemas.py": "a9598adf90edd2eaf5c755965155ab075b4d7174980e452379a9e384963201ce",
 "etl/transform_aqi.py": "04da4ef32208922744fd2568179a40f1a1a6c287925602f01fd05f8834fd2601",
 "etl/transform_caaqms.py": "992fd6013af5a164a86510f56afb26181b1bf3c2bbfcb333fd29d5f4542ed985",
 "etl/validation.py": "06ed4aa59f043f899aeab49758e4758e68fd1330627587f4727b91c8ce1fb96c",
 "etl/verify_full_load.py": "8498f242286b7f2ea1b3b858ac2691fded2a08e2da36f2a9e297e0c7e63fe3e7",
 "models/24h/final/24h_final_model.joblib": "b2069a4b5671cf1ec7e4047922c351514513d5f2b8ccf4256377b5a03bc829f8",
 "models/24h/final/feature_names_final.joblib": "bf11b205a39bcb994b87bb88b47c46df7b794ae4aaafc795b2f826251b84d00b",
 "models/24h/final/imputer_final_24h.joblib": "3ee82708a96a30ccc1da28fe537084eb0659a3c24a2def6ea2eeea855af788c4",
 "models/24h/final/scaler_final_24h.joblib": "bf2ea17accc327515ceb1f499dac38ae49a232bfd80ea299a5b340bbe9ff74ea",
 "models/lgb_model_1h_v1.joblib": "90c9167b22128f23cce46d51a7b163164c1022f25375fadae3b1fa08d0945095",
 "models/lgb_model_6h_v1.joblib": "a0305bd29b496e7fe64a24fb336558f6883e65c4bb5e6ec1159fc5cd2c47e17f",
 "reports/modeling/pm25/pm25_forecast_metrics.csv": "737691770ee30b0e979e5ab5cade27d1c1f40bcd821cd0153f25476a0b1f6839",
 "reports/modeling/probabilistic/probabilistic_metrics.json": "1c124a22f44e5ccad8d230ed8537cc1a3455d43862e07d6a0e927d1ef639927f",
 "requirements-forecast.txt": "cb774172584fb149cf422721fd291c6b4f9be3a03f01d73ee0a729e0439c9160",
 "requirements-live.txt": "f611af1ac0ba9f34956026d419b200bae35b5163e780b751d058211e76e703af",
 "src/analysis/cost_aware_threshold_analysis.py": "0ac85494c4cf82ec741b0a32c38de7cb35e84d9276fdb7a9674b98fb944f125d",
 "src/analysis/diwali_causal_analysis.py": "67d1a6d2352be1e01cfeaa344faac16f74b2baa25df8bb0521746863ab934264",
 "src/eda/advanced_statistical_analysis.py": "ebd5f64a4cc5a1e9db637561665320bb3498fd791b02f5592d913cb6e7c4bfdb",
 "src/eda/data_health_and_coverage.py": "1bfe55d38924b17494d3408ba3d304d786d283f9b3f711d4c9e4d6ebd014d956",
 "src/eda/generate_visualizations.py": "f6642970ea1430e49d2de9a719e0f76a34b9f2d92c24d5903938bb0ccfd8c8c1",
 "src/eda/temporal_and_station_analysis.py": "7991734ce13b248e2b3041222a6293877fd2e6863710cafc49aa48405c64e786",
 "src/evaluation/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
 "src/evaluation/build_metrics_table.py": "e80b35bf6ec233fa1705f7c3eadcea57acce55967868f0af294766a562da0f52",
 "src/evaluation/calibration_side_check.py": "9f074e8ab2ccd69019927adcc7707237eedc9a6bc67edfdd2bbfefc0a04832fd",
 "src/evaluation/feature_availability_audit.py": "5e09130798b72788da153e09a6b88e6a1a187cd9e7d64a1f1e67b99064dafae0",
 "src/evaluation/rolling_origin_backtest.py": "f66ef13df2668353959092162d6ebc9bdeea70b75ec20085f2ffaee0539043d5",
 "src/evaluation/waqi_roundtrip_audit.py": "434b5f02322cd2293040bc71fe73335810f760ea57ebd9f7d2ae5aa0b5179d3b",
 "src/features/build_features.py": "18eee02c45633220ed46f56f112a34f314bce9d2b533bc7109b01badfd62b360",
 "src/features/build_live_matrix.py": "3f3f98adb0414e2143293e21953aa97b43aa86fc6b47ded9a26cad0423c49d8f",
 "src/features/compute_cpcb_aqi.py": "ac8494663ed2a20830e8386a5ea526cda0b2f762cf13775082a297e4182da2a3",
 "src/features/exogenous_drivers.py": "c272e94c107421e339cc4352fcc336dd5bde0179aa58d3b0463a3de687ca77e4",
 "src/features/export_ml_datasets.py": "75587aa730722f7b18c1a01f4806f8c0a9637b2e061209362ae5b46a785c1d01",
 "src/features/target_analysis.py": "34e4de25a69f6ad1db5ae00e55507dbe8cfb5e9341d86d9132b283713cfd8fd1",
 "src/live/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
 "src/live/backfill.py": "04559847d0fef54f5b89f08ae46bad6e580af2e5c4d87d68289873c6edf4ea89",
 "src/live/calibrate_gate.py": "d5b3437dac02451f33c0f2d8d415252fe3b8d9ac994c093ba92c0b0a59a68585",
 "src/live/config.py": "7047a15fb111239cb83485ecfbaa92e5431c4e1a7a005175285f11f00e2fd985",
 "src/live/db.py": "47843df4f3147051aede6397ab1adf0c8174eb42ea9cb0c384f6a7c845985ab8",
 "src/live/epa_aqi.py": "5876a286524a0ba12cf2a2ce0e3b7799be2d2f07359ad3e8b5873333720392c6",
 "src/live/forecast.py": "4d3911685ee48a7b8ff0b78c786afa2ddb4ffdc16cc2437fc7f16458e11f6007",
 "src/live/forecast_store.py": "5036e1aed6629f5221e087a1a9c4c1cf9fcc571f2a99401c936ae0f2bb82ad8b",
 "src/live/ingest.py": "89e0562840f789a6a7ef09a8c2e1f24de4d251956b929ae072caa8d19a7aa791",
 "src/live/load_stations.py": "40da49af76b535379d5a3341f00ab6b2a724a7c9798f7f19fd34b20fb829cf88",
 "src/live/models.py": "e69edae72cb3d3a0e4b2b402a93b3c31ec2e3ac7cba2a2ef382053e15a2da813",
 "src/live/openaq.py": "aca08e13aac63f2effdb5b48242ec17936f1a81fb17ef5b6acea50bc16ec87ef",
 "src/live/openaq_overlap_check.py": "4fd2dde181c10ba141462296352e4d11f30598ea99a9e4aa7c30c1d1b3c6ac83",
 "src/live/parity.py": "cfccef74cd04115c2becbccc14730916ec5c3e365ba79f23f1e61d229e2903bd",
 "src/live/probe_sources.py": "bc3ceee4f1659d9c2e5d69fb1c665b6afbc27bada67c472bc64607f11e36466e",
 "src/live/qc.py": "62b2bd2e654c83e3dafef231937a15357c2e5d103876c2bb5f6a4417212086f1",
 "src/live/sources/__init__.py": "9e27c188045aafe2ff6c73cba8ca122e772bfe09f37195f5c2b682bac188bb69",
 "src/live/sources/base.py": "7802a7cdb4757d44d319db968284f6e3b4df4947792b8edc6a145e4392116602",
 "src/live/sources/data_gov_in.py": "91074a9780bd207a19a670f607ff1730f034b4304c86fb39a57c427915e9dbb8",
 "src/live/sources/openaq_source.py": "2e01ef256a5efd559c75ec8d10c990996663fba7f3e716371b624f6a01e2c935",
 "src/live/store.py": "3a963d3a5d8df6e56bd4f33a44e4ba4e7a64d9f9fb6e4bbb952c5e91de5ee5e4",
 "src/live/waqi_station_check.py": "5de68b1c08c33ee8454eb0f83d8eca9651ddbc188480ed9ddf9f8c82a65f2be9",
 "src/live/weather.py": "fbecbef650552e9ed1a3a7c1aafeb04a633608776104ec0426d4dffbaff4caf7",
 "src/models/ablation_exogenous.py": "9700c51d93c7b32b4853e20e92c1b573dd6c61611769920cc5b070f38ea3330e",
 "src/models/baselines.py": "24ad96879ebdf7d90138a070985f43ce88ae9643c0daf0a97a5b9ff951b99c5d",
 "src/models/evaluate.py": "581ee6b1cdeb003f5690e2385b0f71cf70b2d97d42686414809311ae9604254c",
 "src/models/feature_analysis.py": "466fe42eb339b4d1713e17beec42294089484bfd40388b11658af50e25c1d86b",
 "src/models/live_overrides.py": "fb8f41c07cda3295f4f511355fd32cc46dbe5d59e3967f5428ff1924edfc8dc6",
 "src/models/live_retrain.py": "84c1ae639d8f9a3123767d1e0cc1bb21095e47dc5ddd17be02fefe89bb5615c3",
 "src/models/live_verify_export.py": "cb4eb37715e37186c7c305c2e55597107be54af0c4d145118d47c231ba8296d0",
 "src/models/phase_6c_refinement.py": "1350a0d380040b9a6b8841279076dc6824db444cae5019fbd9aafcfd4e2d34e3",
 "src/models/phase_7_classification.py": "4d5cf33be238d7ff081e282fefaa7afc3f117b4262e5fef5bd188c246ea180fa",
 "src/models/retrain_24h_extended.py": "40545ec98fd2b3a54d620352e894ba2008d55008d8c34d60c8922f4fadffa82f",
 "src/models/splits.py": "f1108734641aa717338f24cdec51f50bc4ba6e81e387647ff5d2d5223d6f04dc",
 "src/models/train_ml_baseline.py": "d877214f4e3566419fbc0de47253d77b6b0c698290cf26aba59c747f1a72c1e1",
 "src/models/train_pm25_forecast.py": "f1b1e0cdec358bbf282ab420c2f8002322594ba118a8ae5acafd25a6368b0268",
 "src/models/train_quantile_forecast.py": "a7712d8d7a98fa6c3d6dc44b154315b41769461fdb774bf6bda2a2b8788420c5",
 "src/models/tuning.py": "ed80604aaecd004675df7998dd8d76feddbc84fbe4797ea7b66cfd7dfad40ec3"
}
def sha(path): return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()
bad = [f for f, h in BUNDLE_SHA256.items() if not pathlib.Path(f).exists() or sha(f) != h]
assert not bad, f'{len(bad)} bundle file(s) missing or changed since the notebook was generated, e.g. {bad[:5]}'
recorded = json.loads(pathlib.Path('config/model_hashes.json').read_text())
frozen_bad = [f for f, h in recorded.items() if sha(f) != h]
assert not frozen_bad, f'frozen model files differ from config/model_hashes.json: {frozen_bad}'
print(f'{len(BUNDLE_SHA256)} bundle files and {len(recorded)} frozen model files verified against recorded hashes')

In [ ]:
# pinned to the versions the forecast job installs (requirements-forecast.txt): exported pickles must load there
PINS = ['scikit-learn==1.9.0', 'lightgbm==4.7.0', 'joblib==1.6.0', 'pyarrow', 'psycopg2-binary', 'python-dotenv', 'requests', 'matplotlib']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-800:])
import sklearn, lightgbm, pandas, numpy, joblib
print('python', platform.python_version(), '| sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__)
assert sklearn.__version__ == '1.9.0' and lightgbm.__version__ == '4.7.0', 'library versions differ from requirements-forecast.txt; do not export models'
print('NOTE: the frozen models were pickled with other versions; unpickle warnings are expected, a crash is not.')

In [ ]:
def run(args, extra_env=None):
    env = {**os.environ, 'PYTHONPATH': str(PROJECT_DIR), 'PYTHONWARNINGS': 'ignore'}
    env.update(extra_env or {})
    r = subprocess.run([sys.executable, '-W', 'ignore', *args], capture_output=True, text=True, env=env)
    print(r.stdout[-7000:]); print(r.stderr[-3000:])
    assert r.returncode == 0, f'step failed ({" ".join(args)}); do not use partial output'

START = ['--start', '2025-07-01'] if SMOKE else []
run(['-m', 'src.features.build_live_matrix', *START])
import pandas as pd
for name in ('features_live', 'features_live_masked'):
    frame = pd.read_parquet(f'data/processed/{name}.parquet')
    assert set(frame.aqi_source.dropna().unique()) == {'recomputed'}, 'matrix must carry recomputed AQI only'
    print(name, frame.shape, frame.timestamp.min(), '..', frame.timestamp.max())

In [ ]:
# SECTION A: AQI models (test window scored once; a second run is refused unless LIVE_RETRAIN_ALLOW_RESCORE=1)
if RUN_AQI:
    run(['-m', 'src.models.live_retrain'], {'LIVE_RETRAIN_SMOKE': '1' if SMOKE else '0'})

In [ ]:
# SECTION B: PM2.5 on live-masked inputs (Tier 2 variant selection on validation, standard profile, exported)
profile = 'smoke' if SMOKE else 'standard'
if RUN_PM25:
    run(['-m', 'src.models.train_pm25_forecast'], {'T2_PROFILE': profile, 'LIVE_MATRIX_PATH': 'data/processed/features_live_masked.parquet',
                                                   'LIVE_OUT_DIR': 'reports/modeling/live_pm25', **({} if SMOKE else {'LIVE_EXPORT_DIR': 'models/live/pm25'})})
# SECTION C: 24h quantile + P(severe)
if RUN_QUANTILE:
    run(['-m', 'src.models.train_quantile_forecast'], {'T2_PROFILE': profile, 'LIVE_MATRIX_PATH': 'data/processed/features_live_masked.parquet',
                                                      'LIVE_OUT_DIR': 'reports/modeling/live_quantile',
                                                      'LIVE_EXPORT_DIR': 'models/live_smoke/quantile' if SMOKE else 'models/live/quantile'})

In [ ]:
if not SMOKE and RUN_AQI and RUN_PM25 and RUN_QUANTILE:
    run(['-m', 'src.models.live_verify_export'])          # re-hashes, reloads and predicts with every exported model; non-zero on any problem
else:
    print('verification skipped (SMOKE or a section was switched off)')

In [ ]:
from IPython.display import Markdown, display
for path in ['reports/modeling/live_retrain/live_retrain_metrics.md', 'reports/modeling/live_pm25/pm25_forecast_metrics.md', 'reports/modeling/live_quantile/quantile_metrics.md']:
    if pathlib.Path(path).exists():
        display(Markdown(pathlib.Path(path).read_text()))
ref = pathlib.Path('reports/modeling/pm25/pm25_forecast_metrics.csv')
if ref.exists():
    print('Tier 2 reference (official-AQI matrix, all inputs) for the PM2.5 table above:')
    display(pd.read_csv(ref)[['horizon', 'split', 'model_mae', 'persistence_mae', 'skill_mae', 'skill_mae_ci_lo', 'skill_mae_ci_hi']])

In [ ]:
assert not SMOKE, 'SMOKE results are never zipped'
import shutil
stage = pathlib.Path('/content/tier3_live_retrain_results') if os.path.exists('/content') else pathlib.Path('tier3_live_retrain_results')
shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True)
for src in ['models/live', 'reports/modeling/live_retrain', 'reports/modeling/live_pm25', 'reports/modeling/live_quantile']:
    if pathlib.Path(src).exists():
        shutil.copytree(src, stage / src)
shutil.make_archive(str(stage), 'zip', stage)
try:
    from google.colab import files; files.download(str(stage) + '.zip')
except ImportError:
    print('results zip written locally:', str(stage) + '.zip')